In [1]:
import pandas as pd
import os
from datetime import datetime

# Carpeta donde están los archivos ZSR460
carpeta = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\ZSR460"

# Carpeta de salida (la misma de siempre)
output_folder = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs"

# Fecha y hora actual para el nombre del archivo
fecha_hora = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_folder, f"ZSR460_limpio_{fecha_hora}.csv")

# Lista para guardar los DataFrames
dfs = []

# Itera sobre todos los archivos en la carpeta
for archivo in os.listdir(carpeta):
    ruta = os.path.join(carpeta, archivo)
    if os.path.isfile(ruta) and archivo.lower().endswith('.xls'):
        # Leer solo la primera fila después del salto para saber cuántas columnas hay
        encabezados = pd.read_csv(ruta, sep='\t', skiprows=9, nrows=1, header=None, encoding='latin1')
        n_cols = encabezados.shape[1]
        # Leer el archivo: saltar 9 filas, usar la primera fila después del salto como encabezados, y leer desde la columna 3 en adelante
        df = pd.read_csv(ruta, sep='\t', skiprows=9, header=0, encoding='latin1', usecols=range(2, n_cols))
        # Quitar espacios al inicio y final de todos los nombres de columnas
        df.columns = [col.strip() for col in df.columns]
        # Eliminar columnas completamente vacías
        df = df.dropna(axis=1, how='all')
        # Eliminar columnas no deseadas si existen
        columnas_a_borrar = ['Un', 'UV', 'AgServTran', 'Chofer', 'PrecioMáx', 'PsEx', 'Solic.', 'Nombre 1.1', 'CP', 'Población', 'Región']
        df = df.drop(columns=[col for col in columnas_a_borrar if col in df.columns], errors='ignore')
        # Eliminar filas donde la primera columna esté vacía
        if not df.empty:
            df = df.dropna(subset=[df.columns[0]])
        # Eliminar filas de encabezados repetidos (si la primera columna es igual al nombre de la columna)
        if not df.empty and isinstance(df.iloc[0,0], str):
            colname = df.columns[0]
            df = df[df[df.columns[0]] != colname]
        # Eliminar filas donde la columna 'Creado el' esté vacía (después de limpiar Transporte)
        if 'Creado el' in df.columns:
            df = df.dropna(subset=['Creado el'])
        # Formatear columnas de fecha 'Creado el' e 'InPlanCarg' de dd.mm.aaaa a dd/mm/aaaa
        for col in ['Creado el', 'InPlanCarg']:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], format='%d.%m.%Y', errors='coerce').dt.strftime('%d/%m/%Y')
        # Quitar comas y convertir a número las columnas 'Peso total' y 'Volumen'
        for col in ['Peso total', 'Volumen']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(',', '', regex=False)
                df[col] = pd.to_numeric(df[col], errors='coerce')
        # Dividir y renombrar columnas
        if 'Peso total' in df.columns:
            df['Peso total(ton)'] = df['Peso total'] / 1000
            df = df.drop(columns=['Peso total'])
        if 'Volumen' in df.columns:
            df['Volumen(m3)'] = df['Volumen'] / 1000000
            df = df.drop(columns=['Volumen'])
        # Asegurar que ciertas columnas sean texto y quitar .0 al final si existe
        columnas_texto = ['Transporte', 'Destinat.', 'Entrega', 'Doc.ventas']
        for col in columnas_texto:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(r'\.0$', '', regex=True)
        df['Archivo'] = archivo  # Agrega columna con el nombre del archivo
        dfs.append(df)

# Une todos los DataFrames
if dfs:
    ZSR460 = pd.concat(dfs, ignore_index=True)
    # Eliminar filas duplicadas solo si son exactamente iguales en todas las columnas
    ZSR460 = ZSR460.drop_duplicates()
    print('Columnas detectadas:', list(ZSR460.columns))
    print(ZSR460.head())
    # Guarda el resultado en CSV con fecha y hora
    ZSR460.to_csv(output_path, index=False, encoding='latin1')
    print(f"✅ Archivo ZSR460 generado y guardado en: {output_path}")
else:
    print("No se encontraron archivos válidos en la carpeta ZSR460.")

Columnas detectadas: ['Transporte', 'Creado el', 'InPlanCarg', 'Nombre 1', 'Tipo', 'Placas', 'Destinat.', 'Nombre 1.2', 'Entrega', 'Doc.ventas', 'Peso total(ton)', 'Volumen(m3)', 'Archivo']
  Transporte   Creado el  InPlanCarg              Nombre 1         Tipo  \
0    2340491  01/05/2026  01/05/2026  ARRENDA TP LOGISTICA  Trailer 48´   
1    2340491  01/05/2026  01/05/2026  ARRENDA TP LOGISTICA  Trailer 48´   
2    2340491  01/05/2026  01/05/2026  ARRENDA TP LOGISTICA  Trailer 48´   
3    2340491  01/05/2026  01/05/2026  ARRENDA TP LOGISTICA  Trailer 48´   
4    2340491  01/05/2026  01/05/2026  ARRENDA TP LOGISTICA  Trailer 48´   

   Placas Destinat.          Nombre 1.2   Entrega Doc.ventas  Peso total(ton)  \
0  12BA5T    510387  Sucursal Monterrey  84897960   10900458         0.526311   
1  12BA5T    510387  Sucursal Monterrey  84897962   10900514         0.438502   
2  12BA5T    510387  Sucursal Monterrey  84897963   68808872         0.010869   
3  12BA5T    510387  Sucursal Monte